# Lab5.4 — Unde se consumă timpul?


Un accelerator rapid nu garantează automat o aplicație rapidă. Separăm costul copierii datelor de costul DMA + PL și de costul readucerii rezultatului în memoria software.


Pregătim Overlay-ul Resize.


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

OVERLAY_DIR = Path("overlay")
OVERLAY_DIR.mkdir(exist_ok=True)
LOCAL_BIT = OVERLAY_DIR / "resizer.bit"
LOCAL_HWH = OVERLAY_DIR / "resizer.hwh"

def find_resizer_overlay():
    if LOCAL_BIT.exists() and LOCAL_HWH.exists():
        return LOCAL_BIT
    try:
        import pynq_helloworld
        pkg = Path(pynq_helloworld.__file__).resolve().parent / "notebooks" / "edge"
        pkg_bit = pkg / "resizer.bit"
        pkg_hwh = pkg / "resizer.hwh"
        if pkg_bit.exists() and pkg_hwh.exists():
            return pkg_bit
    except ImportError:
        pass
    urls = {
        LOCAL_BIT: "https://www.xilinx.com/bin/public/openDownload?filename=pynqhelloworld.resizer.pynqz2.bit",
        LOCAL_HWH: "https://www.xilinx.com/bin/public/openDownload?filename=pynqhelloworld.resizer.pynqz2.hwh",
    }
    for path, url in urls.items():
        if not path.exists():
            print(f"Downloading {path.name} ...")
            urlretrieve(url, path)
    return LOCAL_BIT

RESIZER_BIT = find_resizer_overlay()
print("Overlay:", RESIZER_BIT)

Localizăm imaginea de test.


In [ ]:
from pathlib import Path

IMAGE_PATH = Path("../Lab4/images/fruits.jpg")
if not IMAGE_PATH.exists():
    raise FileNotFoundError("Nu a fost găsită imaginea ../Lab4/images/fruits.jpg")
print("Image:", IMAGE_PATH)

Încărcăm imaginea și inițializăm acceleratorul și bufferele.


In [ ]:
import cv2
import numpy as np
import time
from pynq import Overlay, allocate

image_bgr = cv2.imread(str(IMAGE_PATH), cv2.IMREAD_COLOR)
old_h, old_w, _ = image_bgr.shape
new_w, new_h = 320, 240

overlay = Overlay(str(RESIZER_BIT))
dma = overlay.axi_dma_0
resizer = overlay.resize_accel_0
in_buffer = allocate(shape=image_bgr.shape, dtype=np.uint8)
out_buffer = allocate(shape=(new_h, new_w, 3), dtype=np.uint8)

resizer.register_map.src_rows = old_h
resizer.register_map.src_cols = old_w
resizer.register_map.dst_rows = new_h
resizer.register_map.dst_cols = new_w

Definim separat operația DMA + accelerator, pentru a o putea cronometra independent.


In [ ]:
def run_kernel():
    dma.sendchannel.transfer(in_buffer)
    dma.recvchannel.transfer(out_buffer)
    resizer.write(0x00, 0x81)
    dma.sendchannel.wait()
    dma.recvchannel.wait()

Executăm o rulare de încălzire înaintea benchmark-ului.


In [ ]:
np.copyto(in_buffer, image_bgr)
run_kernel()
_ = np.array(out_buffer)
print("Warm-up completed.")

Măsurăm separat copierea în bufferul DMA, execuția DMA+PL și copierea rezultatului într-un array NumPy obișnuit.


In [ ]:
REPEATS = 30
copy_in_times = []
kernel_times = []
copy_out_times = []

for _ in range(REPEATS):
    t0 = time.perf_counter()
    np.copyto(in_buffer, image_bgr)
    t1 = time.perf_counter()
    run_kernel()
    t2 = time.perf_counter()
    result = np.array(out_buffer)
    t3 = time.perf_counter()
    copy_in_times.append(t1 - t0)
    kernel_times.append(t2 - t1)
    copy_out_times.append(t3 - t2)

t_copy_in = np.mean(copy_in_times)
t_kernel = np.mean(kernel_times)
t_copy_out = np.mean(copy_out_times)
t_total = t_copy_in + t_kernel + t_copy_out

print(f"Copy input : {t_copy_in*1e3:8.3f} ms")
print(f"PL + DMA   : {t_kernel*1e3:8.3f} ms")
print(f"Copy output: {t_copy_out*1e3:8.3f} ms")
print(f"Total      : {t_total*1e3:8.3f} ms")
print(f"FPS total  : {1/t_total:8.2f}")

## Interpretare

`PL + DMA` nu reprezintă doar timpul aritmetic al nucleului Resize; include și transferurile DMA. `Total` aproximează mai bine costul unei etape reale dintr-o aplicație Edge.


Eliberăm bufferele după benchmark.


In [ ]:
del in_buffer, out_buffer


## Întrebări

1. Care componentă domină timpul total?
2. De ce nu putem numi `PL + DMA` „timpul pur al FPGA-ului”?
3. Cum am putea reduce costul copierilor în aplicații reale?
